# Vision I — from pixels to a trainable image classifier

Companion to [the complete lecture](../../vision1.html). Work in order: predict, calculate, then run.

**Three settings:** hand-chosen four-patch arithmetic; a full small ViT trained on generated noisy images; a pretrained ViT processing real Oxford-IIIT Pet photographs. Keep their parameters and claims separate.

Needs NumPy and PyTorch. The real-image scripts additionally need timm and Pillow, and download a public checkpoint if it is not cached. Nothing plays audio.


In [1]:
from pathlib import Path
import json
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
# Works when opened from the repository root, src/, or notebooks/vision/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/vision1_worksheet.py").exists())


## 1. Every worksheet parameter
First inspect W_patch and both heads. Which coordinates do the nonzero entries select?

In [2]:
"""Four visible patches, two attention heads, and one reproducible worksheet.

Parameters are chosen for arithmetic, not fitted. The two named arrangements
are the exercise; this is not a general horizontal/vertical recognizer.
"""
from pathlib import Path
import json
import numpy as np


def parameters():
    root2 = float(np.sqrt(2))
    return {
        'd_model': 4, 'd_k': 2, 'd_v': 2,
        'W_patch': [[.25, 0, 0, 0]] * 4,
        'b_patch': [0, 0, 0, 1],
        'cls': [0, 0, 0, 1],
        'positions': [[0, 0, 0, 0], [0, 0, 0, 0],
                      [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 1, 0]],
        'heads': [
            {'W_Q': [[0, 0], [0, 0], [0, 0], [1, 1]],
             'W_K': [[root2, 0], [0, root2], [0, 0], [0, 0]],
             'W_V': [[1, 0], [0, 1], [0, 0], [0, 0]]},
            {'W_Q': [[0, 0], [0, 0], [0, 0], [1, 1]],
             'W_K': [[root2, 0], [0, 0], [0, root2], [0, 0]],
             'W_V': [[1, 0], [0, 0], [0, 1], [0, 0]]},
        ],
        'W_O': [[1, 0, 1, 0], [0, 1, 0, 0],
                [-1, 0, 1, 0], [0, 0, 0, 1]],
        'W_class': [[-4, 4], [0, 0], [0, 0], [0, 0]],
        'classes': ['Across the top', 'Down the left'],
        'images': {
            'horizontal': [[1, 1, 1, 1], [1, 1, 1, 1],
                           [0, 0, 0, 0], [0, 0, 0, 0]],
            'vertical': [[1, 1, 0, 0], [1, 1, 0, 0],
                         [1, 1, 0, 0], [1, 1, 0, 0]],
        },
    }


def softmax(x):
    ex = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return ex / ex.sum(axis=-1, keepdims=True)


def forward(image, use_positions=True):
    p = parameters()
    pixels = np.asarray(image, dtype=float)
    # Four row-major patches; each patch's pixels are TL, TR, BL, BR.
    patches = pixels.reshape(2, 2, 2, 2).transpose(0, 2, 1, 3).reshape(4, 4)
    content = patches @ np.asarray(p['W_patch']) + p['b_patch']
    E = np.vstack([p['cls'], content])
    if use_positions:
        E = E + p['positions']
    heads = []
    for w in p['heads']:
        Q, K, V = [E @ np.asarray(w[key]) for key in ['W_Q', 'W_K', 'W_V']]
        scores = Q @ K.T / np.sqrt(p['d_k'])
        A = softmax(scores)
        H = A @ V
        heads.append(dict(Q=Q, K=K, V=V, scores=scores, A=A, H=H,
                          contributions=A[0, :, None] * V))
    joined = np.concatenate([h['H'] for h in heads], axis=-1)
    delta = joined @ np.asarray(p['W_O'])
    updated = E + delta
    logits = updated[0] @ np.asarray(p['W_class'])
    probability = softmax(logits)
    return dict(pixels=pixels, patches=patches, content=content, E=E,
                heads=heads, joined=joined, delta=delta, updated=updated,
                logits=logits, probability=probability)


def serializable(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, dict):
        return {k: serializable(v) for k, v in x.items()}
    if isinstance(x, list):
        return [serializable(v) for v in x]
    return x


def export():
    p = parameters()
    p['cases'] = {
        name + ('_positions' if positions else '_no_positions'):
        serializable(forward(image, positions))
        for name, image in p['images'].items() for positions in [True, False]
    }
    path = Path(__file__).with_name('vision1-worksheet.json')
    path.write_text(json.dumps(p, indent=2) + '\n')
    return p




In [3]:
p = parameters()
for key in ['W_patch', 'b_patch', 'positions', 'W_O', 'W_class']:
    print(key, np.array(p[key]), sep='\n')
for number, head in enumerate(p['heads'], 1):
    print('Head', number)
    for key, value in head.items(): print(key, np.array(value), sep='\n')


W_patch
[[0.25 0.   0.   0.  ]
 [0.25 0.   0.   0.  ]
 [0.25 0.   0.   0.  ]
 [0.25 0.   0.   0.  ]]
b_patch
[0 0 0 1]
positions
[[0 0 0 0]
 [0 0 0 0]
 [0 0 1 0]
 [0 1 0 0]
 [0 1 1 0]]
W_O
[[ 1  0  1  0]
 [ 0  1  0  0]
 [-1  0  1  0]
 [ 0  0  0  1]]
W_class
[[-4  4]
 [ 0  0]
 [ 0  0]
 [ 0  0]]
Head 1
W_Q
[[0 0]
 [0 0]
 [0 0]
 [1 1]]
W_K
[[1.41421356 0.        ]
 [0.         1.41421356]
 [0.         0.        ]
 [0.         0.        ]]
W_V
[[1 0]
 [0 1]
 [0 0]
 [0 0]]
Head 2
W_Q
[[0 0]
 [0 0]
 [0 0]
 [1 1]]
W_K
[[1.41421356 0.        ]
 [0.         0.        ]
 [0.         1.41421356]
 [0.         0.        ]]
W_V
[[1 0]
 [0 0]
 [0 1]
 [0 0]]


## 2. Trace the entire forward pass
Calculate P1’s projected row, one score, and one weighted value before revealing the output.

In [4]:
r = forward(p['images']['horizontal'])
for key in ['pixels','patches','content','E']:
    print(key, np.round(r[key], 6), sep='\n')
for i, head in enumerate(r['heads'], 1):
    print('HEAD', i)
    for key in ['Q','K','V','scores','A','contributions','H']:
        print(key, np.round(head[key], 6), sep='\n')
for key in ['joined','delta','updated','logits','probability']:
    print(key, np.round(r[key], 6), sep='\n')
print('Loss', -np.log(r['probability'][0]))


pixels
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
patches
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
content
[[1. 0. 0. 1.]
 [1. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]]
E
[[0. 0. 0. 1.]
 [1. 0. 0. 1.]
 [1. 0. 1. 1.]
 [0. 1. 0. 1.]
 [0. 1. 1. 1.]]
HEAD 1
Q
[[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]]
K
[[0.       0.      ]
 [1.414214 0.      ]
 [1.414214 0.      ]
 [0.       1.414214]
 [0.       1.414214]]
V
[[0. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [0. 1.]]
scores
[[0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]]
A
[[0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]]
contributions
[[0.       0.      ]
 [0.228944 0.      ]
 [0.228944 0.      ]
 [0.       0.228944]
 [0.       0.228944]]
H
[[0.457888 0.457888]
 [0.457888 0.457888]


## 3. Independent PyTorch comparison
Pack the two heads into MultiheadAttention. PyTorch Linear stores output-by-input weights, hence the transposes.

In [5]:
layer = nn.MultiheadAttention(4, 2, bias=False, batch_first=True,
                              dtype=torch.float64).eval()
with torch.no_grad():
    packed = np.concatenate([np.concatenate([h[k] for h in p['heads']], axis=1).T
                              for k in ['W_Q','W_K','W_V']], axis=0)
    layer.in_proj_weight.copy_(torch.tensor(packed))
    layer.out_proj.weight.copy_(torch.tensor(np.array(p['W_O']).T))
    E = torch.tensor(r['E']).unsqueeze(0)
    delta, weights = layer(E,E,E,average_attn_weights=False)
np.testing.assert_allclose(delta[0], r['delta'], atol=1e-12)
for h in range(2):
    np.testing.assert_allclose(weights[0,h],r['heads'][h]['A'],atol=1e-12)
print('Both head matrices and the projected update agree to 1e-12.')


Both head matrices and the projected update agree to 1e-12.


## 4. Position control
Predict all four results. Moving contents to different positions differs from reordering complete positioned rows.

In [6]:
for name, pixels in p['images'].items():
    for positions in [True,False]:
        result = forward(pixels,positions)
        print(name, 'positions:',positions, 'P(class):',result['probability'])
        if not positions: np.testing.assert_allclose(result['probability'],[.5,.5])


horizontal positions: True P(class): [0.85703513 0.14296487]
horizontal positions: False P(class): [0.5 0.5]
vertical positions: True P(class): [0.14296487 0.85703513]
vertical positions: False P(class): [0.5 0.5]


## 5. One learning step that fits on the board
Freeze the worksheet; add a zero class bias and update only it using SGD with learning rate 0.5.

In [7]:
logits = torch.tensor(r['logits'], dtype=torch.float64)
bias = torch.zeros(2, dtype=torch.float64, requires_grad=True)
loss = F.cross_entropy((logits+bias)[None],torch.tensor([0]))
loss.backward()
expected = r['probability']-np.array([1.,0.])
np.testing.assert_allclose(bias.grad,expected,atol=1e-12)
print('Gradient',bias.grad.numpy())
with torch.no_grad(): bias -= .5*bias.grad
print('New bias',bias.detach().numpy())
print('P(class)',(logits+bias).softmax(-1).detach().numpy())
print('Loss before/after',float(loss.detach()),float(F.cross_entropy((logits+bias)[None],torch.tensor([0])).detach()))


Gradient [-0.14296487  0.14296487]
New bias [ 0.07148244 -0.07148244]
P(class) [0.87367437 0.12632563]
Loss before/after 0.15427637415539522 0.13504754446086184


## 6. Restore the rest of the block
These independent examples explain LayerNorm and the row MLP; they are not inserted into the earlier hand calculation.

In [8]:
z = torch.tensor([1.,0.,0.,1.], dtype=torch.float64)
print('LayerNorm', F.layer_norm(z,(4,),eps=1e-5).numpy())
x = torch.tensor([1.,-1.],dtype=torch.float64)
W1 = torch.tensor([[1.,0.,1.],[0.,1.,1.]],dtype=torch.float64)
W2 = torch.tensor([[1.,0.],[0.,1.],[0.,0.]],dtype=torch.float64)
hidden = F.gelu(x@W1,approximate='none')
print('GELU hidden',hidden.numpy())
print('MLP message',(hidden@W2).numpy())
print('Residual',(x+hidden@W2).numpy())


LayerNorm [ 0.99998 -0.99998 -0.99998  0.99998]
GELU hidden [ 0.84134475 -0.15865525  0.        ]
MLP message [ 0.84134475 -0.15865525]
Residual [ 1.84134475 -1.15865525]


## 7. A complete small ViT
The following classes are exactly those used in the actual training experiment. Inspect both residual paths and the final CLS readout.

In [9]:
class Block(nn.Module):
    def __init__(self, d=16, heads=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=0, batch_first=True)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 2*d), nn.GELU(), nn.Linear(2*d, d))

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attn(z, z, z, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))

class SmallViT(nn.Module):
    def __init__(self, positions=True, d=16):
        super().__init__()
        self.patch = nn.Conv2d(1, d, kernel_size=2, stride=2)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.zeros(1, 17, d), requires_grad=positions)
        self.use_positions = positions
        self.blocks = nn.Sequential(Block(d), Block(d))
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, 2)
        nn.init.normal_(self.cls, std=.02)
        if positions: nn.init.normal_(self.pos, std=.02)

    def forward(self, images):
        x = self.patch(images).flatten(2).transpose(1, 2)  # B,16,16
        x = torch.cat([self.cls.expand(x.size(0), -1, -1), x], dim=1)
        if self.use_positions: x = x + self.pos
        x = self.norm(self.blocks(x))
        return self.head(x[:, 0])  # raw class logits


def dataset(pairs, seed):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for _ in range(pairs):
        row = int(rng.integers(0, 4))
        patches = rng.normal(.10, .07, (4, 4, 2, 2))
        patches[row] += rng.uniform(.60, .85)
        patches = np.clip(patches, 0, 1)
        # Transpose patch LOCATIONS, keeping pixels inside each patch unchanged.
        for grid, label in [(patches, 0), (patches.transpose(1, 0, 2, 3), 1)]:
            images.append(grid.transpose(0, 2, 1, 3).reshape(8, 8))
            labels.append(label)
    return torch.tensor(np.array(images)[:, None], dtype=torch.float32), torch.tensor(labels)



In [10]:
torch.manual_seed(7)
model = SmallViT(positions=True)
images, labels = dataset(2, seed=99)
logits = model(images)
F.cross_entropy(logits, labels).backward()
for name, parameter in model.named_parameters():
    assert parameter.grad is not None and torch.isfinite(parameter.grad).all(),name
print('Input',tuple(images.shape),'logits',tuple(logits.shape))
print('Parameters',sum(p.numel() for p in model.parameters()))
print('Gradients reach every trainable parameter.')


Input (4, 1, 8, 8) logits (4, 2)
Parameters 4882
Gradients reach every trainable parameter.


## 8. Conv2d really is the shared patch projection
Check flatten-and-linear against the convolution with exactly the same weights. Unfold uses channel-major pixel order.

In [11]:
conv = nn.Conv2d(1,16,kernel_size=2,stride=2)
x = torch.randn(3,1,8,8)
patch_rows = F.unfold(x,kernel_size=2,stride=2).transpose(1,2)
a = patch_rows @ conv.weight.flatten(1).T + conv.bias
b = conv(x).flatten(2).transpose(1,2)
torch.testing.assert_close(a,b)
print('Conv2d == shared Linear on unfolded patches:',tuple(a.shape))


Conv2d == shared Linear on unfolded patches: (3, 16, 16)


## 9. Reproduce the trained checkpoint results

The training script already ran 80 epochs on 512 training images. Validation selected a checkpoint; these saved weights are evaluated below on the independently generated test split.

To rerun training from initialization:
```bash
python notebooks/vision/train_small_vit.py
```
This overwrites only the two saved small-model checkpoints and their report. The script includes the optimizer, minibatches, validation selection and final test evaluation. Test results below are **one-seed synthetic results**, not a benchmark of natural-image ability.


In [12]:
test_images, test_labels = dataset(128, seed=33)
for positions in [True,False]:
    fitted = SmallViT(positions).eval()
    filename = 'small-vit-positions.pt' if positions else 'small-vit-no-positions.pt'
    fitted.load_state_dict(torch.load(ROOT/'notebooks/vision'/filename,weights_only=True,map_location='cpu'))
    with torch.inference_mode():
        logits = fitted(test_images)
        correct = int((logits.argmax(-1)==test_labels).sum())
        paired_gap = float((logits.softmax(-1)[0::2]-logits.softmax(-1)[1::2]).abs().max())
    print('positions:',positions,'correct:',correct,'/ 256','max paired probability gap:',paired_gap)
    assert correct == (256 if positions else 128)


positions: True correct: 256 / 256 max paired probability gap: 0.9994640946388245
positions: False correct: 128 / 256 max paired probability gap: 5.960464477539063e-08


## 10. Real photographs, real measurements

The lecture uses the exact two saved Oxford-IIIT Pet images and the ImageNet checkpoint `timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k`. Preprocessing comes from that checkpoint. The 14×14 maps show raw source probabilities for a specified block, head and query; CLS-source mass is reported separately.

Run the scripts to regenerate real measurements:
```bash
uv run --with timm --with pillow python notebooks/vision/run_real_images.py
uv run --with timm --with pillow python notebooks/vision/inspect_real_vit.py
```
The fixed quadrant interventions fill with normalized RGB zero. They measure sensitivity on this input, not causal importance or dataset accuracy.


In [13]:
results = json.loads((ROOT/'figures/vision1/real-inference.json').read_text())
for result in results['results']: print(result['image_id'],result['top3'])
inspection = json.loads((ROOT/'figures/vision1/inspection.json').read_text())
for record in inspection['attention']:
    total = np.sum(record['patch_weights'])+record['cls_weight']
    np.testing.assert_allclose(total,1.,atol=2e-6)
print('Every recorded attention row sums to one including CLS.')
for result in inspection['occlusion']:
    print(result['region'],result['target_probability'])


newfoundland_31 [{'index': 256, 'label': 'Newfoundland, Newfoundland dog', 'probability': 0.9572675228118896}, {'index': 244, 'label': 'Tibetan mastiff', 'probability': 0.01625724323093891}, {'index': 226, 'label': 'briard', 'probability': 0.006742686033248901}]
Persian_98 [{'index': 283, 'label': 'Persian cat', 'probability': 0.96707683801651}, {'index': 332, 'label': 'Angora, Angora rabbit', 'probability': 0.012567228637635708}, {'index': 728, 'label': 'plastic bag', 'probability': 0.004300369881093502}]
Every recorded attention row sums to one including CLS.
Top left 0.8299893736839294
Top right 0.796161413192749
Bottom left 0.8453509211540222
Bottom right 0.8215695023536682


## 11. Predict first; then run the answers

1. Scores after scaling are `[ln 2, 0]`, values are `[2,0]` and `[0,3]`. What arrives?
2. For a 128×128 RGB image, P=16, D=64 and four heads, find N, Q and A shapes.
3. Halve patch width at fixed resolution. How do token count and score count change?
4. Why does permuting already-positioned rows preserve CLS, while moving image contents can change it?
5. Why does a bright attention cell not alone explain a class prediction?


In [14]:
weights = softmax(np.array([np.log(2),0]))
print('Weights',weights,'message',weights@np.array([[2,0],[0,3]]))
N = (128//16)**2+1
print('E:',(N,64),'Q/head:',(N,16),'A/head:',(N,N))
for P in [32,16,8]:
    N=(224//P)**2+1
    print('Patch size',P,'tokens',N,'scores/head',N*N)
print('Position answer: keep content-position associations when reordering complete rows.')
print('Interpretation answer: values, output projection, residuals and later layers also affect logits.')


Weights [0.66666667 0.33333333] message [1.33333333 1.        ]
E: (65, 64) Q/head: (65, 16) A/head: (65, 65)
Patch size 32 tokens 50 scores/head 2500
Patch size 16 tokens 197 scores/head 38809
Patch size 8 tokens 785 scores/head 616225
Position answer: keep content-position associations when reordering complete rows.
Interpretation answer: values, output projection, residuals and later layers also affect logits.


## References and attribution

- [Original ViT paper](https://arxiv.org/abs/2010.11929)
- [D2L: Vision Transformer](https://d2l.ai/chapter_attention-mechanisms-and-transformers/vision-transformer.html)
- [UvA: Vision Transformer notebook](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial15/Vision_Transformer.html)
- [Stanford CS231n 2025, Lecture 8](https://cs231n.stanford.edu/slides/2025/lecture_8.pdf)
- [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/) via the [timm mirror](https://huggingface.co/datasets/timm/oxford-iiit-pet); image attribution and provenance are in figures/vision1/images.json. Images retain CC BY-SA 4.0 attribution and their owners' copyright.

The diagrams and hand worksheet are original to this teaching series. [Parts I](../../part1.html), [II](../../attention.html) and [III](../../part3.html) provide the earlier examples and notation.
